In [ ]:
# import sys
# from pathlib import Path

# PROJECT_ROOT = Path.cwd().resolve().parent

# if str(PROJECT_ROOT) not in sys.path:
#     sys.path.insert(0, str(PROJECT_ROOT))

# from src.context_builder import build_expanded_fashion_context
# from src.query_generator import generate_search_queries

In [ ]:
# import json

# with open(
#     PROJECT_ROOT / "data" / "context.json",
#     "r",
#     encoding="utf-8"
# ) as file:
#     context = json.load(file)

In [7]:
# Reading User and Clothing Requirement Context
# context = load_context("../data/context.json") Can also use this function from ../src/context_builder
import json

with open("../data/context.json", "r", encoding="utf-8") as file:
    context = json.load(file)

print(context)

{'destination': 'Seoul', 'season': 'winter', 'weather_clothing_requirements': {'outerwear': ['winter coat', 'puffer jacket'], 'top': ['sweater', 'knitwear'], 'bottom': ['long trousers'], 'footwear': ['closed shoes', 'boots'], 'properties': ['warm', 'insulated', 'layerable', 'weather-resistant']}, 'activity_style_requirements': {'sightseeing': ['casual', 'comfortable'], 'dinner': ['smart casual', 'polished']}, 'user_style': ['minimalist']}


In [8]:
from itertools import product

wcr = context["weather_clothing_requirements"]

tops = wcr.get("top", [])
bottoms = wcr.get("bottom", [])
footwear = wcr.get("footwear", [])

combinations = list(product(tops, bottoms, footwear))

for combination in combinations:
    print(combination)

('sweater', 'long trousers', 'closed shoes')
('sweater', 'long trousers', 'boots')
('knitwear', 'long trousers', 'closed shoes')
('knitwear', 'long trousers', 'boots')


In [9]:
def generate_search_queries(context):

    destination = context["destination"].strip()
    season = context["season"].strip()

    wcr = context.get("weather_clothing_requirements", {})
    asr = context.get("activity_style_requirements", {})
    user_styles = context.get("user_style", [])

    queries = []

    # 1. General user-style query
    if user_styles:
        style_text = " ".join(style.strip() for style in user_styles if style.strip())
        if style_text:
            queries.append(f"{destination} {season} {style_text} outfit")

    # 2. Activity-specific queries
    for activity, styles in asr.items():
        style_text = " ".join(style.strip() for style in styles if style.strip())
        queries.append(f"{destination} {season} {activity.strip()} {style_text} outfit")

    # 3. Weather-based outerwear queries
    outerwear = wcr.get("outerwear", [])
    for garment in outerwear:
        if garment.strip():
            queries.append(f"{destination} {season} {garment.strip()} outfit")

    # 4. Clothing combination query
    from itertools import product

    tops = wcr.get("top", [])
    bottoms = wcr.get("bottom", [])
    footwear = wcr.get("footwear", [])

    if tops and bottoms and footwear:
        for top, bottom, shoe in product(tops, bottoms, footwear):
            query = f"{destination} {season} {top} {bottom} {shoe} outfit"
            queries.append(query)

    # 5. Clean whitespace and remove duplicate queries and return
    queries = [" ".join(query.split()) for query in queries]
    return list(dict.fromkeys(queries))

In [13]:
search_queries=generate_search_queries(context)
for query in search_queries:
    print(query)

Seoul winter minimalist outfit
Seoul winter sightseeing casual comfortable outfit
Seoul winter dinner smart casual polished outfit
Seoul winter winter coat outfit
Seoul winter puffer jacket outfit
Seoul winter sweater long trousers closed shoes outfit
Seoul winter sweater long trousers boots outfit
Seoul winter knitwear long trousers closed shoes outfit
Seoul winter knitwear long trousers boots outfit


In [ ]:
# Flickr -- Require PRO Version (Paid Version Required for API)
# Unsplash - Registerd for Developer (Got 50 Requests / Hour)
'''
Unsplash Application - IRS - Fashion Inspiration Retrieval
A non-commercial academic prototype developed as part of a university Intelligent Reasoning Systems project. 
The application aims to retrieve fashion inspiration photographs based on travel destination, season, weather-related clothing requirements, activities, and user style preferences.

The retrieved photographs will be used for visual inspiration and relevance ranking using a pretrained FashionCLIP vision-language model. 
The application will preserve photographer attribution and source links and follow Unsplash API usage requirements. 
It is intended solely for educational research and demonstration, not commercial deployment or redistribution of photographs.
'''


In [ ]:
# expanded_fashion_context = build_expanded_fashion_context(context)

# search_queries = generate_search_queries(context)

# print("Expanded Fashion Context:")
# print(expanded_fashion_context)

# print("\nGenerated Search Queries:")

# for i, query in enumerate(search_queries, start=1):
#     print(f"{i}. {query}")

In [ ]:
# output_dir = PROJECT_ROOT / "outputs"
# output_dir.mkdir(exist_ok=True)

# output_path = output_dir / "search_queries.json"

# with open(output_path, "w", encoding="utf-8") as file:
#     json.dump(search_queries, file, indent=4, ensure_ascii=False)

# print("Search queries saved:", output_path)

Commented the Whole System Integration Scripts Alignment, will take care during integration part

In [14]:
# Loading the Unsplash Key

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

UNSPLASH_ACCESS_KEY = os.getenv("UNSPLASH_ACCESS_KEY")

if not UNSPLASH_ACCESS_KEY:
    raise ValueError("Unsplash Access Key not found.")

print("Unsplash Access Key loaded successfully.")

Unsplash Access Key loaded successfully.


In [15]:
print(search_queries[0])

Seoul winter minimalist outfit


In [23]:
query = search_queries[0]

In [31]:
# Unsplash search request
query = search_queries[0]
import requests

url = "https://api.unsplash.com/search/photos"

headers = {
    "Authorization": f"Client-ID {UNSPLASH_ACCESS_KEY}",
    "Accept-Version": "v1"
}

params = {
    "query": query,
    "per_page": 10,
    "page": 1
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=20
)

print("Status Code:", response.status_code)
print(
    "Remaining Requests:",
    response.headers.get("X-Ratelimit-Remaining")
)

response.raise_for_status()

data = response.json()

Status Code: 200
Remaining Requests: 49


In [32]:
## Saving the response, because I got limited API Call Limit (In real time system workflow saving is not required)

import json
from pathlib import Path

cache_dir = Path("../data/image_api_calls")
cache_dir.mkdir(parents=True, exist_ok=True)

cache_path = cache_dir / "unsplash_search_test.json"

with open(cache_path, "w", encoding="utf-8") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

print("API response saved:", cache_path)

API response saved: ..\data\image_api_calls\unsplash_search_02.json


In [ ]:
## Saving the Load Code in Comments (For Later if required)
# Load previously saved API response when needed
# with open(cache_path, "r", encoding="utf-8") as file:
#     data = json.load(file)
# print("Photos returned:", len(data["results"]))

In [33]:
# Inspect the retreived photos

photos = data["results"]
print("Number of retrieved photos:", len(photos))

Number of retrieved photos: 10


In [34]:
# Structure of Photo received from API Call
if photos:
    print(photos[0].keys())

dict_keys(['id', 'alternative_slugs', 'created_at', 'updated_at', 'promoted_at', 'width', 'height', 'color', 'blur_hash', 'description', 'breadcrumbs', 'urls', 'likes', 'liked_by_user', 'bookmarked', 'current_user_collections', 'sponsorship', 'topic_submissions', 'asset_type', 'slug', 'alt_description', 'short_description', 'links', 'user'])


In [35]:
# Extracting the fields we need

retrieved_images = []
for photo in photos:

    image_record = {
        "image_id": photo["id"],
        "image_url": photo["urls"]["small"],
        "source_url": photo["links"]["html"],
        "description": (photo.get("description") or photo.get("alt_description")),
        "photographer": photo["user"]["name"],
        "photographer_url": photo["user"]["links"]["html"],
        "search_query": query
    }

    retrieved_images.append(image_record)

print("Extracted image records:", len(retrieved_images))

Extracted image records: 10


In [36]:
for image in retrieved_images[:3]:
    print("Image ID:", image["image_id"])
    print("Description:", image["description"])
    print("Photographer:", image["photographer"])
    print("Image URL:", image["image_url"])
    print("-" * 50)

Image ID: rXANayuznKI
Description: woman standing beside plants
Photographer: Eno Eltora
Image URL: https://images.unsplash.com/flagged/photo-1575238529271-87ca5ee24ae5?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=M3wxMDk2MzE3fDB8MXxzZWFyY2h8MXx8U2VvdWwlMjB3aW50ZXIlMjBtaW5pbWFsaXN0JTIwb3V0Zml0fGVufDB8fHx8MTc5MTQ1MTc2Mnww&ixlib=rb-4.1.0&q=80&w=400
--------------------------------------------------
Image ID: cdlrjH_8g48
Description: a watch, lipstick, and watch band laying on a bed
Photographer: AROMATEEC
Image URL: https://images.unsplash.com/photo-1635866127591-9cb87bebbb91?crop=entropy&cs=tinysrgb&fit=max&fm=jpg&ixid=M3wxMDk2MzE3fDB8MXxzZWFyY2h8Mnx8U2VvdWwlMjB3aW50ZXIlMjBtaW5pbWFsaXN0JTIwb3V0Zml0fGVufDB8fHx8MTc5MTQ1MTc2Mnww&ixlib=rb-4.1.0&q=80&w=400
--------------------------------------------------
Image ID: eVtwC7k7BR4
Description: a woman walking down a street past a brick building
Photographer: Ratapan Anantawat
Image URL: https://images.unsplash.com/photo-1672480757704-c36c0aa9fa

In [37]:
# Display 1st Retreived Image for search query : Seoul winter minimalist outfit
from IPython.display import display, HTML

image = retrieved_images[0]

display(HTML(f"""
    <img src="{image['image_url']}" style="max-width:300px; max-height:400px; object-fit:contain;">
"""))

In [38]:
from IPython.display import display, HTML
from html import escape

for rank, image in enumerate(retrieved_images, start=1):

    image_url = escape(image["image_url"], quote=True)
    photographer = escape(image["photographer"])
    photographer_url = escape(
        image["photographer_url"], quote=True
    )
    source_url = escape(image["source_url"], quote=True)

    display(HTML(f"""
        <div style="margin-bottom: 25px;">
            <p><b>Result {rank}</b></p>

            <img src="{image_url}"
                 style="max-width:300px; max-height:400px;
                        object-fit:contain;">

            <p>
                Photo by
                <a href="{photographer_url}">{photographer}</a>
                on <a href="{source_url}">Unsplash</a>
            </p>
        </div>
    """))

In [ ]:
## Out of 10, around 7 images are fashion outfit related to the query

In [40]:
# Retrieving results for each query (all the query combinations generated before from the user context)
all_retrieved_images = []
# count=1
for query in search_queries:

    params = {
        "query": query,
        "per_page": 10,
        "page": 1
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=20
    )

    response.raise_for_status()
    data = response.json()

    '''
    import json
    from pathlib import Path

    cache_dir = Path("../data/image_api_calls")
    cache_dir.mkdir(parents=True, exist_ok=True)

    cache_path = cache_dir / f"unsplash_search_{count}.json"

    with open(cache_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print("API response saved:", cache_path)
    count+=1
    '''

    photos = data["results"]

    print(f"{query}: {len(photos)} images")

    for photo in photos:

        image_record = {
            "image_id": photo["id"],
            "image_url": photo["urls"]["small"],
            "source_url": photo["links"]["html"],
            "description": (photo.get("description") or photo.get("alt_description")),
            "photographer": photo["user"]["name"],
            "photographer_url": photo["user"]["links"]["html"],
            "search_query": query
        }

        all_retrieved_images.append(image_record)

print("Total records:", len(all_retrieved_images))

API response saved: ..\data\image_api_calls\unsplash_search_1.json
Seoul winter minimalist outfit: 10 images
API response saved: ..\data\image_api_calls\unsplash_search_2.json
Seoul winter sightseeing casual comfortable outfit: 10 images
API response saved: ..\data\image_api_calls\unsplash_search_3.json
Seoul winter dinner smart casual polished outfit: 8 images
API response saved: ..\data\image_api_calls\unsplash_search_4.json
Seoul winter winter coat outfit: 1 images
API response saved: ..\data\image_api_calls\unsplash_search_5.json
Seoul winter puffer jacket outfit: 3 images
API response saved: ..\data\image_api_calls\unsplash_search_6.json
Seoul winter sweater long trousers closed shoes outfit: 1 images
API response saved: ..\data\image_api_calls\unsplash_search_7.json
Seoul winter sweater long trousers boots outfit: 3 images
API response saved: ..\data\image_api_calls\unsplash_search_8.json
Seoul winter knitwear long trousers closed shoes outfit: 6 images
API response saved: ..\dat

In [42]:
print("Remaining Requests:", response.headers.get("X-Ratelimit-Remaining"))

Remaining Requests: 41


In [43]:
# Remove Duplicates
unique_images = {}

for image in all_retrieved_images:
    image_id = image["image_id"]

    if image_id not in unique_images:
        unique_images[image_id] = image

retrieved_images = list(unique_images.values())

print("Before deduplication:", len(all_retrieved_images))
print("After deduplication:", len(retrieved_images))

Before deduplication: 43
After deduplication: 19


In [44]:
# Display the 19 images got from the multiple combined search queries

from IPython.display import display, HTML
from html import escape

for rank, image in enumerate(retrieved_images, start=1):

    image_url = escape(image["image_url"], quote=True)
    photographer = escape(image["photographer"])
    photographer_url = escape(
        image["photographer_url"], quote=True
    )
    source_url = escape(image["source_url"], quote=True)

    display(HTML(f"""
        <div style="margin-bottom: 25px;">
            <p><b>Result {rank}</b></p>

            <img src="{image_url}"
                 style="max-width:300px; max-height:400px;
                        object-fit:contain;">

            <p>
                Photo by
                <a href="{photographer_url}">{photographer}</a>
                on <a href="{source_url}">Unsplash</a>
            </p>
        </div>
    """))

In [45]:
total_records = len(all_retrieved_images)
unique_count = len(retrieved_images)

duplicate_count = total_records - unique_count
duplicate_rate = duplicate_count / total_records

print("Total retrieved:", total_records)
print("Unique images:", unique_count)
print("Duplicates removed:", duplicate_count)
print(f"Duplicate rate: {duplicate_rate:.2%}")

Total retrieved: 43
Unique images: 19
Duplicates removed: 24
Duplicate rate: 55.81%


In [ ]:
## Further from the retrieved Images (19 Images), around 15-16 Images are relevant to the query combinations.